# Tập 2.5 · Dựng 3D từ ảnh chụp (Structure from Motion) 🏠

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap25/tap25_sfm.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 2 · ⏱ ~50 phút · Giáo trình: Principles of Robot Autonomy, [Chương 7](https://porabook.com/book/stereo-vision/) (Mục 7.3)

Một camera đi vòng quanh ngôi nhà, 12 tấm ảnh dựng lại 402 điểm 3D. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video (vài giây).
2. **Tự viết** bước hai ảnh đầu: ma trận cơ bản F, tam giác hoá, chọn nghiệm (R, t) (3 bài), có ô chấm tự động.
3. Xem ngôi nhà dựng lại trước và sau bundle adjustment.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap25")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import sfm as goc                  # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"{len(goc.P_THAT)} điểm trên nhà · {goc.SO_ANH} ảnh")
print(f"Hai ảnh đầu: {goc.N_CHUNG} cặp điểm chung · 4 nghiệm (R, t) có số điểm nằm trước cả hai camera: {goc.DEM_4}")
print(f"Sai số chiếu lại: ghép tuần tự {goc.RMS_TRUOC:.2f} px -> sau bundle adjustment {goc.RMS_SAU:.2f} px")
print(f"Sai số 3D (sau khi gắn tỉ lệ bằng cạnh {goc.RONG} m): {goc.LOI0.mean() * 100:.1f} cm -> {goc.LOI1.mean() * 100:.1f} cm")

## 2. Bài tập

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `tim_F(p1, p2)` | thuật toán 8 điểm có chuẩn hoá, ép hạng 2: $p_1^T F p_2 = 0$ |
| **2** | `tam_giac(P1, P2, x1, x2)` | mỗi cặp điểm 4 phương trình $a_u P_1^{(3)} - P_1^{(1)}, \dots$ → SVD |
| **3** | `dem_truoc_camera(R, t, X)` | số điểm có độ sâu dương ở cả hai camera |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np


def chuan_hoa(p):
    """Có sẵn: ma trận 3×3 dời tâm đám điểm về gốc, co giãn cho khoảng cách trung bình = √2."""
    m = p.mean(0)
    s = np.sqrt(2) / np.linalg.norm(p - m, axis=1).mean()
    return np.array([[s, 0, -s * m[0]], [0, s, -s * m[1]], [0, 0, 1]])


def tim_F(p1, p2):
    """Bài 1 (Eq. 7.5–7.6, thuật toán 8 điểm): ma trận cơ bản F thoả  p1ᵀ·F·p2 = 0  cho mọi cặp điểm tương ứng.

    1. Chuẩn hoá: T1 = chuan_hoa(p1), T2 = chuan_hoa(p2); a = T1·[u, v, 1]ᵀ (ảnh 1), b = T2·[u, v, 1]ᵀ (ảnh 2).
    2. Mỗi cặp cho 1 dòng của ma trận W: np.outer(a, b).ravel()  (9 số).
    3. F_n = véc-tơ kỳ dị nhỏ nhất của W (np.linalg.svd(W)[2][-1]), reshape(3, 3).
    4. Ép hạng 2: U, S, Vt = svd(F_n); F_n = U @ diag(S[0], S[1], 0) @ Vt.
    5. Bỏ chuẩn hoá: F = T1ᵀ @ F_n @ T2."""
    raise NotImplementedError("TODO Bài 1: thuật toán 8 điểm")


def tam_giac(P1, P2, x1, x2):
    """Bài 2 (Eq. 7.8): tam giác hoá. Mỗi cặp pixel (a ở ảnh 1, b ở ảnh 2) cho 4 phương trình theo X = (X, Y, Z, 1):

        a_u·P1[2] − P1[0],   a_v·P1[2] − P1[1],   b_u·P2[2] − P2[0],   b_v·P2[2] − P2[1]      (mỗi dòng 4 số)
    Xếp thành A (4×4), X = véc-tơ kỳ dị nhỏ nhất của A, rồi chia cho X[3]. Trả về mảng (N, 3) các điểm 3D."""
    raise NotImplementedError("TODO Bài 2: tam giác hoá")


def dem_truoc_camera(R, t, X):
    """Bài 3: E tách ra được 4 nghiệm (R, t); chỉ nghiệm đúng cho điểm nằm TRƯỚC cả hai camera.

    Camera 1 ở gốc: độ sâu z1 = X[:, 2]. Camera 2: độ sâu z2 = (X·Rᵀ + t)[:, 2].
    Trả về số điểm có z1 > 0 VÀ z2 > 0 (kiểu int)."""
    raise NotImplementedError("TODO Bài 3: đếm điểm trước cả hai camera")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
ve(None, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

SfM không biết tỉ lệ: nhân cả mô hình lên k lần thì sai số chiếu lại có đổi không?

In [ ]:
for k in [0.5, 1.0, 3.0]:
    cams = [(R, t * k) for R, t in goc.CAMS1]
    diem = {j: P * k for j, P in goc.DIEM1.items()}
    print(f"phóng mô hình × {k}: sai số chiếu lại {goc.sai_chieu_lai(cams, diem):.4f} px")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap25).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)